# SEPO Stage 2 — GRPO Training
**Model**: Gemma 3 4B (SFT warm-start checkpoint)  
**Game**: Iterated Prisoner's Dilemma  
**Objective**: `J(π) = payoff - 0.5×exploitability - 2.0×collusion - 1.0×externality`

Runtime: **T4 GPU** (16GB) — use `Runtime > Change runtime type > T4`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate peft bitsandbytes huggingface_hub numpy

In [ ]:
# ── Cell 3: HuggingFace login (paste your token when prompted) ────────────────
from huggingface_hub import login
login()

In [ ]:
# ── Cell 4: Clone repo (grpo-stage2 branch) ───────────────────────────────────
# Generate a GitHub token at: github.com/settings/tokens (classic, repo scope)
import os
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"  # paste your token

!git clone -b grpo-stage2 https://{GITHUB_TOKEN}@github.com/kirankumarmanku/sepo.git
%cd sepo

In [ ]:
# ── Cell 5: Verify GPU memory ─────────────────────────────────────────────────
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 6: IPD Game + SEPO reward (self-contained, no import needed) ─────────
import re
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Tuple

R, T, S, P = 3, 5, 0, 1
COOPERATE, DEFECT = 0, 1
ACTION_LABEL = {COOPERATE: "<SILENT>", DEFECT: "<TESTIFY>"}
ACTION_PARSE = {
    "<SILENT>": COOPERATE, "<TESTIFY>": DEFECT,
    "SILENT": COOPERATE,   "TESTIFY": DEFECT,
    "cooperate": COOPERATE, "defect": DEFECT,
}
PAYOFF_TABLE = {
    (COOPERATE, COOPERATE): (R, R),
    (DEFECT,    COOPERATE): (T, S),
    (COOPERATE, DEFECT):    (S, T),
    (DEFECT,    DEFECT):    (P, P),
}
N_ROUNDS = 8

SYSTEM_PROMPT = """You are playing the Iterated Prisoner's Dilemma game.

Rules:
- Each round you choose one of two actions: <SILENT> or <TESTIFY>
- If both players choose <SILENT>: you each get 3 points
- If you choose <TESTIFY> and opponent chooses <SILENT>: you get 5, opponent gets 0
- If you choose <SILENT> and opponent chooses <TESTIFY>: you get 0, opponent gets 5
- If both choose <TESTIFY>: you each get 1 point

Your goal is to maximise your total score over all rounds.
Respond with ONLY your action: <SILENT> or <TESTIFY>. Nothing else."""

class AlwaysCooperate:
    name = "always-cooperate"
    def act(self, my_h, opp_h, rng): return COOPERATE

class AlwaysDefect:
    name = "always-defect"
    def act(self, my_h, opp_h, rng): return DEFECT

class TitForTat:
    name = "tit-for-tat"
    def act(self, my_h, opp_h, rng): return opp_h[-1] if opp_h else COOPERATE

class GenerousTitForTat:
    name = "generous-tit-for-tat"
    def act(self, my_h, opp_h, rng):
        if not opp_h: return COOPERATE
        return COOPERATE if (opp_h[-1] == COOPERATE or rng.random() < 0.25) else DEFECT

class GrimTrigger:
    name = "grim-trigger"
    def act(self, my_h, opp_h, rng): return DEFECT if DEFECT in opp_h else COOPERATE

TRAIN_POOL     = [AlwaysDefect, TitForTat, GenerousTitForTat]
EXPLOITER_POOL = [AlwaysDefect]
COLLUSIVE_POOL = [AlwaysDefect]

def build_user_prompt(t, n_rounds, h_llm, h_opp):
    lines = [f"Round {t + 1} of {n_rounds}."]
    if h_llm:
        lines.append("Your past actions:       " + ", ".join(ACTION_LABEL[a] for a in h_llm))
        lines.append("Opponent's past actions: " + ", ".join(ACTION_LABEL[a] for a in h_opp))
        my_sc  = sum(PAYOFF_TABLE[(m, o)][0] for m, o in zip(h_llm, h_opp))
        opp_sc = sum(PAYOFF_TABLE[(m, o)][1] for m, o in zip(h_llm, h_opp))
        lines.append(f"Scores so far — you: {my_sc}, opponent: {opp_sc}")
    else:
        lines.append("This is the first round. No history yet.")
    lines.append("\nWhat is your action?")
    return "\n".join(lines)

def parse_action(text):
    text = text.strip()
    for token, action in ACTION_PARSE.items():
        if token in text: return action
    return None

print("IPD environment ready.")

In [ ]:
# ── Cell 7: Load model (4-bit) + LoRA + frozen reference model ────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

MODEL_ID = "kartiinx/gemma-3-4b-sepo-sft"

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Strip any stale quantization_config from MLX export before loading
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL_ID)
if hasattr(cfg, "quantization_config"):
    del cfg.quantization_config

# Training model — 4bit base + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=cfg,
    quantization_config=bnb_4bit,
    device_map="auto",
    ignore_mismatched_sizes=True,
)
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Reference model — 4bit, fully frozen
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=cfg,
    quantization_config=bnb_4bit,
    device_map="auto",
    ignore_mismatched_sizes=True,
)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

device = torch.device("cuda")
print(f"\nVRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 8: Episode runner + log prob recomputation ───────────────────────────
import torch.nn.functional as F

@torch.no_grad()
def run_episode(model, tokenizer, opponent_cls, seed, temperature=0.8):
    rng = np.random.default_rng(seed)
    opponent = opponent_cls()
    h_llm, h_opp = [], []
    actions, opp_actions, payoffs, opp_payoffs = [], [], [], []
    all_input_ids, all_gen_ids = [], []

    for t in range(N_ROUNDS):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(t, N_ROUNDS, h_llm, h_opp)},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        ).to(device)

        out = model.generate(
            input_ids, max_new_tokens=8,
            do_sample=(temperature > 0), temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_ids = out[0, input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        action = parse_action(gen_text) or COOPERATE

        opp_action = opponent.act(h_opp, h_llm, rng)
        pay, opp_pay = PAYOFF_TABLE[(action, opp_action)]

        h_llm.append(action); h_opp.append(opp_action)
        actions.append(action); opp_actions.append(opp_action)
        payoffs.append(float(pay)); opp_payoffs.append(float(opp_pay))
        all_input_ids.append(input_ids); all_gen_ids.append(gen_ids)

    return {
        "actions": actions, "opp_actions": opp_actions,
        "payoffs": payoffs, "opp_payoffs": opp_payoffs,
        "h_llm": h_llm, "h_opp": h_opp,
        "input_ids": all_input_ids, "gen_ids": all_gen_ids,
    }


def recompute_log_probs(model, input_ids_list, gen_ids_list):
    log_probs = []
    for input_ids, gen_ids in zip(input_ids_list, gen_ids_list):
        full_ids = torch.cat([input_ids[0], gen_ids], dim=0).unsqueeze(0)
        logits = model(full_ids).logits[0]
        n_in = input_ids.shape[1]
        n_gen = gen_ids.shape[0]
        pred_logits = logits[n_in - 1: n_in - 1 + n_gen]
        lp = F.log_softmax(pred_logits, dim=-1)
        log_probs.append(lp[torch.arange(n_gen), gen_ids].sum())
    return log_probs

print("Episode runner ready.")

In [ ]:
# ── Cell 9: SEPO reward ───────────────────────────────────────────────────────
# Lambda values from paper (sepo_gtbench_ipd_results.md):
#   λe=3.6, λc=3.2, λx=2.4
LAMBDA_E, LAMBDA_C, LAMBDA_X = 3.6, 3.2, 2.4

def compute_sepo_reward(episodes_by_pool):
    train_eps    = episodes_by_pool["train"]
    exploit_eps  = episodes_by_pool["exploiter"]
    collusive_eps = episodes_by_pool["collusive"]

    utility = float(np.mean([np.mean(e["payoffs"]) for e in train_eps]))

    if exploit_eps:
        llm_pay = np.mean([np.mean(e["payoffs"]) for e in exploit_eps])
        opp_pay = np.mean([np.mean(e["opp_payoffs"]) for e in exploit_eps])
        exploitability = float(max(0.0, opp_pay - llm_pay))
    else:
        exploitability = 0.0

    if collusive_eps:
        collusion = float(np.mean([
            sum(1 for a in e["h_llm"] if a == COOPERATE) / N_ROUNDS
            for e in collusive_eps
        ]))
    else:
        collusion = 0.0

    shortfalls = [
        max(0.0, (R - (PAYOFF_TABLE[(a, o)][0] + PAYOFF_TABLE[(a, o)][1]) / 2) / R)
        for e in train_eps + exploit_eps
        for a, o in zip(e["h_llm"], e["h_opp"])
    ]
    externality = float(np.mean(shortfalls)) if shortfalls else 0.0

    reward = utility - LAMBDA_E * exploitability - LAMBDA_C * collusion - LAMBDA_X * externality
    return reward, {"utility": utility, "exploitability": exploitability,
                    "collusion": collusion, "externality": externality, "sepo_reward": reward}

print("SEPO reward ready.")

In [ ]:
# ── Cell 10: GRPO training loop ───────────────────────────────────────────────
import json
from pathlib import Path

N_GROUPS   = 2    # opponents per step (keep low for T4 memory)
N_ROLLOUTS = 4    # rollouts per group (G)
TEMPERATURE = 0.8
BETA        = 0.01
LR          = 1e-5
N_ITERS     = 300
SAVE_EVERY  = 50
OUTPUT_DIR  = Path("/content/grpo_gemma3_ipd")
OUTPUT_DIR.mkdir(exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)

log = []
pools_config = {
    "train":     TRAIN_POOL,
    "exploiter": EXPLOITER_POOL,
    "collusive": COLLUSIVE_POOL,
}

print(f"Starting GRPO — {N_ITERS} steps, G={N_ROLLOUTS}, groups={N_GROUPS}")
print(f"SEPO: λe={LAMBDA_E}  λc={LAMBDA_C}  λx={LAMBDA_X}\n")

for step in range(N_ITERS):
    optimizer.zero_grad()
    all_log_probs, all_ref_lps, all_advantages = [], [], []
    step_metrics = []

    for g in range(N_GROUPS):
        rollout_rewards, rollout_data = [], []

        for r in range(N_ROLLOUTS):
            seed_base = step * 10000 + g * 1000 + r * 100
            episodes_by_pool = {k: [] for k in pools_config}
            train_ids = []

            for pool_name, pool in pools_config.items():
                for opp_cls in pool:
                    ep = run_episode(model, tokenizer, opp_cls,
                                     seed=seed_base + hash(opp_cls.name) % 97,
                                     temperature=TEMPERATURE)
                    episodes_by_pool[pool_name].append(ep)
                    if pool_name == "train":
                        train_ids.append((ep["input_ids"], ep["gen_ids"]))

            reward, metrics = compute_sepo_reward(episodes_by_pool)
            rollout_rewards.append(reward)
            rollout_data.append((train_ids, metrics))

        rewards = np.array(rollout_rewards, dtype=np.float32)
        adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8) \
              if rewards.std() > 1e-8 else np.zeros_like(rewards)

        for r, (train_ids, metrics) in enumerate(rollout_data):
            A = float(adv[r])
            for inp_ids, gen_ids in train_ids:
                lps = recompute_log_probs(model, inp_ids, gen_ids)
                with torch.no_grad():
                    ref_lps = recompute_log_probs(ref_model, inp_ids, gen_ids)
                for lp, ref_lp in zip(lps, ref_lps):
                    all_log_probs.append(lp)
                    all_ref_lps.append(ref_lp.detach())
                    all_advantages.append(A)
            step_metrics.append(metrics)

    if not all_log_probs:
        continue

    lp_t  = torch.stack(all_log_probs)
    rlp_t = torch.stack(all_ref_lps)
    adv_t = torch.tensor(all_advantages, device=device, dtype=lp_t.dtype)

    pg_loss = -(adv_t * lp_t).mean()
    kl      = (lp_t - rlp_t).mean()
    loss    = pg_loss + BETA * kl

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    optimizer.step()

    avg = {k: float(np.mean([m[k] for m in step_metrics])) for k in step_metrics[0]}
    avg["kl"] = float(kl.detach())
    log.append({"step": step, "loss": float(loss), **avg})

    if step % 10 == 0:
        print(f"Step {step:3d} | loss={float(loss):.4f} | "
              f"u={avg['utility']:.2f} | e={avg['exploitability']:.2f} | "
              f"c={avg['collusion']:.2f} | x={avg['externality']:.2f} | "
              f"sepo={avg['sepo_reward']:.2f} | kl={avg['kl']:.4f}")

    if step % SAVE_EVERY == 0 and step > 0:
        ckpt = OUTPUT_DIR / f"step_{step:03d}"
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        with open(OUTPUT_DIR / "log.json", "w") as f:
            json.dump(log, f, indent=2)
        print(f"  → Saved {ckpt}")

# Final save
model.save_pretrained(OUTPUT_DIR / "final")
tokenizer.save_pretrained(OUTPUT_DIR / "final")
with open(OUTPUT_DIR / "log.json", "w") as f:
    json.dump(log, f, indent=2)
print("\nDone!")

In [ ]:
# ── Cell 11: Plot training curves ─────────────────────────────────────────────
import matplotlib.pyplot as plt

steps   = [e["step"] for e in log]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("GRPO + SEPO Training — Gemma 3 4B / IPD")

for ax, key, title in zip(
    axes.flat,
    ["loss", "sepo_reward", "utility", "exploitability", "collusion", "externality"],
    ["Loss", "SEPO Reward", "Utility (payoff)", "Exploitability", "Collusion", "Externality"],
):
    ax.plot(steps, [e[key] for e in log])
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 12: Upload final model to HuggingFace ────────────────────────────────
# Create a new private HF repo first: kartiinx/gemma-3-4b-sepo-grpo
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=str(OUTPUT_DIR / "final"),
    repo_id="kartiinx/gemma-3-4b-sepo-grpo",
    repo_type="model",
)
print("Uploaded to kartiinx/gemma-3-4b-sepo-grpo")